In [57]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split , GridSearchCV , KFold , cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score , mean_squared_error , mean_absolute_error
from sklearn.compose import ColumnTransformer , TransformedTargetRegressor
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler , OneHotEncoder , OrdinalEncoder , PowerTransformer
from sklearn.base import BaseEstimator , TransformerMixin
import optuna
from lightgbm import LGBMRegressor
import mlflow
import dagshub
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate , plot_slice

In [58]:
dagshub.init(repo_owner='mridul0010', repo_name='food-delivery-time-prediction', mlflow=True)

Accessing as mridul0010

Initialized MLflow to track repo "mridul0010/food-delivery-time-prediction"

Repository mridul0010/food-delivery-time-prediction initialized!

In [59]:
mlflow.set_tracking_uri("https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow")

In [60]:
mlflow.set_experiment("1. Hyperparameter Tuning - LGBM")

<Experiment: artifact_location='mlflow-artifacts:/343667758e5c41ba824a6fef0f390496', creation_time=1783244373727, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783244373727, lifecycle_stage='active', name='1. Hyperparameter Tuning - LGBM', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [61]:
df = pd.read_csv('../data/Cleaned Delivery Dataset.csv')

In [62]:
pd.set_option('display.max_columns' , None)

In [63]:
df.drop(columns=['Delivery_Agent'] , inplace=True)

In [64]:
df['Order_Datetime'] = pd.to_datetime(df['Order_Datetime'])
df['Pickup_Datetime'] = pd.to_datetime(df['Pickup_Datetime'])

In [65]:
class FeatureEngineering(BaseEstimator, TransformerMixin):
    
    def __init__(self):
        self.rating_bins = None

    def fit(self, X, y=None):
        X = X.copy()
        
        # Store bins for consistent transformation
        self.rating_bins = pd.qcut(
            X['Delivery_person_Ratings'],
            q=3,
            retbins=True,
            duplicates='drop'
        )[1]
        
        return self

    def transform(self, X):
        X = X.copy()

        # Datetime conversion
        X['Order_Datetime'] = pd.to_datetime(X['Order_Datetime'], errors='coerce')
        X['Pickup_Datetime'] = pd.to_datetime(X['Pickup_Datetime'], errors='coerce')

        # Rating group
        X["delivery_rating_group"] = pd.cut(
            X['Delivery_person_Ratings'],
            bins=self.rating_bins,
            labels=['Low', 'Medium', 'High'],
            include_lowest=True
        )

        # Age group
        X["age_group"] = pd.cut(
            X['Delivery_person_Age'],
            bins=[14, 25, 35, 60],
            labels=['Young', 'Adult', 'Senior']
        )

        # Distance group
        X["distance_group"] = pd.cut(
            X['distance_km'],
            bins=[0, 5, 10, 25],
            labels=['Short Distance', 'Medium Distance', 'Long Distance']
        )

        # Time features
        X['Prep_Time(min)'] = (
            X['Pickup_Datetime'] - X['Order_Datetime']
        ).dt.total_seconds() / 60

        X['Order_hour'] = X['Order_Datetime'].dt.hour
        X['Order_day'] = X['Order_Datetime'].dt.day_name()

        X['isWeekend'] = X['Order_day'].isin(["Saturday", "Sunday"]).astype(int)

        X['Time_Of_Day'] = pd.cut(
            X['Order_hour'],
            bins=[0, 6, 12, 18, 24],
            labels=["Night", "Morning", "Afternoon", "Evening"],
            include_lowest=True
        )

        # Drop raw datetime
        X = X.drop(columns=['Order_Datetime', 'Pickup_Datetime'], errors='ignore')

        return X

    def get_feature_names_out(self , input_features = None):
        return input_features

In [66]:
X = df.drop(columns=['Time_taken (min)'])
y = df['Time_taken (min)']

In [67]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

In [68]:
X

,City,Zone,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,distance_km,Order_Datetime,Pickup_Datetime,Weather_conditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City_Type
0,Dehradun,Zone17,36,4.2,30.327968,78.046106,30.397968,78.116106,10.28,2022-12-02 21:55:00,2022-12-02 22:10:00,Fog,Jam,Good,Snack,motorcycle,3,No,Metropolitian
1,Kochi,Zone16,21,4.7,10.003064,76.307589,10.043064,76.347589,6.24,2022-02-13 14:55:00,2022-02-13 15:05:00,Stormy,High,Average,Meal,motorcycle,1,No,Metropolitian
2,Pune,Zone13,23,4.7,18.562450,73.916619,18.652450,74.006619,13.79,2022-04-03 17:30:00,2022-04-03 17:40:00,Sandstorms,Medium,Average,Drinks,scooter,1,No,Metropolitian
3,Ludhiana,Zone15,34,4.3,30.899584,75.809346,30.919584,75.829346,2.93,2022-02-13 09:20:00,2022-02-13 09:30:00,Sandstorms,Low,poor,Buffet,motorcycle,0,No,Metropolitian
4,Kanpur,Zone14,24,4.7,26.463504,80.372929,26.593504,80.502929,19.40,2022-02-14 19:50:00,2022-02-14 20:05:00,Fog,Jam,Average,Snack,scooter,1,No,Metropolitian
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39637,Bangalore,Zone16,28,4.9,13.029198,77.570997,13.059198,77.600997,4.66,2022-03-30 21:55:00,2022-03-30 22:05:00,Sandstorms,Jam,Average,Meal,scooter,1,No,Metropolitian
39638,Ranchi,Zone16,35,4.2,23.371292,85.327872,23.481292,85.437872,16.60,2022-08-03 21:45:00,2022-08-03 21:55:00,Windy,Jam,Good,Drinks,motorcycle,1,No,Metropolitian
39639,Chennai,Zone08,30,4.9,13.022394,80.242439,13.052394,80.272439,4.66,2022-11-03 23:50:00,2022-11-04 00:05:00,Cloudy,Low,Average,Drinks,scooter,0,No,Metropolitian
39640,Coimbatore,Zone11,20,4.7,11.001753,76.986241,11.041753,77.026241,6.23,2022-07-03 13:35:00,2022-07-03 13:40:00,Cloudy,High,poor,Snack,motorcycle,1,No,Metropolitian


In [69]:
numerical = ['Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude',
       'Restaurant_longitude', 'Delivery_location_latitude',
       'Delivery_location_longitude', 
       'multiple_deliveries', 'distance_km', 'Prep_Time(min)', 'Order_hour',
       'isWeekend']

ohe_col = ['City', 'Zone', 'Weather_conditions',
       'Type_of_order', 'Type_of_vehicle',
       'City_Type','Order_day', 'Time_Of_Day']

ordinal_col = ['Road_traffic_density' , 'Vehicle_condition' , 'Festival',
       'delivery_rating_group' , 'age_group' , 'distance_group']

In [70]:
Road_traffic_density = ['Low', 'Medium', 'High', 'Jam']
Vehicle_condition = ['poor', 'Average', 'Good' , 'Excellent']
Festival = ['No', 'Yes']
delivery_rating_group = ['Low', 'Medium', 'High']
age_group = ['Young', 'Adult', 'Senior']         
distance_group = ['Short Distance', 'Medium Distance', 'Long Distance']

In [71]:
transformer = ColumnTransformer(
    transformers=[
        ("ohe" , OneHotEncoder(sparse_output=False , handle_unknown='ignore') , ohe_col),
        ('oe' , OrdinalEncoder(categories=[
            Road_traffic_density , Vehicle_condition,
            Festival , delivery_rating_group,
            age_group , distance_group
        ]) , ordinal_col),
        ('Scaling' , StandardScaler() , numerical)
    ],remainder='passthrough'
)

In [72]:
pipeline = Pipeline(
    [
        ("FeatureEngineering" , FeatureEngineering()),
        ("ColumnTransformer" , transformer)
    ]
) 

In [73]:
X_train_trans = pipeline.fit_transform(X_train)
X_test_trans = pipeline.transform(X_test)

In [74]:
pt = PowerTransformer()

In [75]:
def objective(trial):
    with mlflow.start_run(nested=True):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 300),
            "max_depth": trial.suggest_int("max_depth", 1, 40),
            "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.2),
            
            # Subsample (bagging fraction) requires subsample_freq to be active
            "subsample": trial.suggest_float("subsample", 0.4, 1.0),
            "subsample_freq": 1, 
            
            # Highly recommended for LightGBM to control overfitting
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
            
            "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 10.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 100.0),
            
            # Keeps the output clean during Optuna trials
            "verbose": -1, 
            
            "random_state": 42,
            "n_jobs": -1,
        }

        # log params
        mlflow.log_params(params)


        lgbm = LGBMRegressor(**params)
        model = TransformedTargetRegressor(regressor=lgbm , transformer=pt)

        model.fit(X_train_trans , y_train)

        cv_score = cross_val_score(
            model ,
            X_train_trans , 
            y_train , 
            cv=10 , 
            scoring="neg_mean_absolute_error",
            n_jobs=-1
        )

        mean_score = -(cv_score.mean())

        # log avg cross val error
        mlflow.log_metric("cross_val_error" , mean_score)

        return mean_score

In [76]:
study = optuna.create_study(direction='minimize')

with mlflow.start_run(run_name="best_model"):
    # optimize the objective function
    study.optimize(objective , n_trials=50 , n_jobs=-1 , show_progress_bar=True)

    # log the best params
    mlflow.log_params(study.best_params)

    # log best score
    mlflow.log_metric("best_score" , study.best_value)

    # training the lgbm on best param
    best_lgbm = LGBMRegressor(**study.best_params)

    best_model = TransformedTargetRegressor(regressor=best_lgbm , transformer=pt)

    best_model.fit(X_train_trans , y_train)

    y_pred_train = best_model.predict(X_train_trans)
    y_pred_test = best_model.predict(X_test_trans)

    scores = cross_val_score(
        best_model,
        X_train_trans,
        y_train,
        scoring="neg_mean_absolute_error",
        cv=5,n_jobs=-1
    )

    # logging metrics
    mlflow.log_metric("Training_error" ,mean_absolute_error(y_train ,y_pred_train))
    mlflow.log_metric("Test_error" ,mean_absolute_error(y_test ,y_pred_test))
    mlflow.log_metric("Training_r2" ,r2_score(y_train ,y_pred_train))
    mlflow.log_metric("Test_r2" ,r2_score(y_test ,y_pred_test))
    mlflow.log_metric("cross_val" , -scores.mean())

    # Generate the optuna plots
    fig_history = plot_optimization_history(study)
    fig_parallel = plot_parallel_coordinate(study)
    fig_importance = plot_param_importances(study)
    fig_slice = plot_slice(study)

    # Loginf plots
    mlflow.log_figure(fig_history, "optuna_plots/optimization_history.html")
    mlflow.log_figure(fig_importance, "optuna_plots/param_importances.html")
    mlflow.log_figure(fig_parallel, "optuna_plots/parallel_coordinate.html")
    mlflow.log_figure(fig_slice, "optuna_plots/plot_slice.html")

    # log the best model 
    mlflow.sklearn.log_model(
        sk_model=best_model, 
        name="model_LGBM",
        serialization_format="cloudpickle"
    )

[I 2026-07-05 16:06:48,820] A new study created in memory with name: no-name-f4e6ca31-66ed-4261-9e9e-3a4de4231fe9
  0%|          | 0/50 [00:00<?, ?it/s]

🏃 View run awesome-cat-994 at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/1/runs/64b9b7258dbe41288a834dbf4f12fa3b
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/1
🏃 View run omniscient-crane-20 at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/1/runs/90f98346ef274654b686b3b983574817
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/1


Best trial: 1. Best value: 3.3589:   2%|▏         | 1/50 [00:30<25:01, 30.64s/it]

[I 2026-07-05 16:07:20,011] Trial 1 finished with value: 3.3589019291406315 and parameters: {'n_estimators': 189, 'max_depth': 6, 'learning_rate': 0.07801249784053041, 'subsample': 0.8723443916224762, 'min_child_samples': 65, 'min_split_gain': 6.4242310197797785, 'reg_lambda': 12.059275844661643}. Best is trial 1 with value: 3.3589019291406315.
🏃 View run brawny-chimp-682 at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/1/runs/f9d7c459553a4b2ea3685e22cffdb663
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/1
🏃 View run secretive-shark-884 at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/1/runs/8e8f8aacb6a34d7d9a02454fe5456c1f
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/1
🏃 View run placid-shark-338 at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/1/runs/a20c884a0

Best trial: 1. Best value: 3.3589:   4%|▍         | 2/50 [00:42<15:44, 19.68s/it]

...

[I 2026-07-05 16:13:06,406] Trial 46 finished with value: 3.288773190429142 and parameters: {'n_estimators': 212, 'max_depth': 11, 'learning_rate': 0.03608552050094978, 'subsample': 0.811346032040532, 'min_child_samples': 23, 'min_split_gain': 2.217345201773576, 'reg_lambda': 24.216284253524922}. Best is trial 18 with value: 3.0949088713493658.


Best trial: 18. Best value: 3.09491: 100%|██████████| 50/50 [06:20<00:00,  7.60s/it]


[I 2026-07-05 16:13:09,449] Trial 47 finished with value: 3.2687769696374636 and parameters: {'n_estimators': 270, 'max_depth': 24, 'learning_rate': 0.10007576643042225, 'subsample': 0.8019435613856807, 'min_child_samples': 21, 'min_split_gain': 1.444203385759283, 'reg_lambda': 23.042292080711125}. Best is trial 18 with value: 3.0949088713493658.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001775 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1512
[LightGBM] [Info] Number of data points in the train set: 31713, number of used features: 87
[LightGBM] [Info] Start training from score 0.000000


C:\Users\LeoML\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\LeoML\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
2026/07/05 16:13:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run best_model at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/1/runs/cb20548ca6884368aebdd2a215784728
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/1


In [77]:
study.best_value

3.0949088713493658

In [78]:
study.best_params

{'n_estimators': 295,
 'max_depth': 40,
 'learning_rate': 0.1119381474660663,
 'subsample': 0.7060775062281114,
 'min_child_samples': 6,
 'min_split_gain': 0.0173687282004773,
 'reg_lambda': 32.39556957788049}

In [79]:
best_lgbm = LGBMRegressor(**study.best_params)

model = TransformedTargetRegressor(regressor=best_lgbm , transformer=pt)

model.fit(X_train_trans , y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002038 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1512
[LightGBM] [Info] Number of data points in the train set: 31713, number of used features: 87
[LightGBM] [Info] Start training from score 0.000000


,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",LGBMRegressor...0775062281114)
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",PowerTransformer()
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",None
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[<U9](87,)","['Column_0','Column_1','Column_2',...,'Column_84','Column_85','Column_86']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,87
regressor_ regressor_: objectFitted regressor.,LGBMRegressor,LGBMRegressor...0775062281114)
transformer_ transformer_: objectTransformer used in :meth:`fit` and :meth:`predict`.,PowerTransformer,PowerTransformer()
,max_depth,40


In [80]:
y_pred = model.predict(X_test_trans)

r2_score(y_test , y_pred)

C:\Users\LeoML\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


0.8228228718681243

In [81]:
fig_history = plot_optimization_history(study)
fig_parallel = plot_parallel_coordinate(study)
fig_importance = plot_param_importances(study)
fig_slice = plot_slice(study)

In [82]:
fig_history

In [83]:
fig_parallel.show()

In [84]:
fig_importance.show()

In [85]:
fig_slice.show()